# Notebook 50 - Is the calibration advantage of pruned detectors a temperature effect?

**Why this notebook.** On CICIoT2023 the dense reference is overconfident (confidence above accuracy by 0.22 on the shallow network and 0.27 on the deep one after two epochs) and the pruned students carry a lower ECE in 71 of 80 paired cells (notebooks 40 and 46). A reviewer will ask the obvious question: one scalar temperature, fitted on held-out rows, removes most overconfidence of a classifier. If it removes the dense model's, is anything left of the students' advantage? The answer decides how the paper states its calibration result.

**What it does.** GPU, about forty minutes. The committed notebook 40 90% structures for magnitude, Taylor and Fisher and the dense teacher, both networks, recovered for two full epochs with notebook 46's seeds (601, 701, 809), recalibrated exactly as in notebook 46. For every model one temperature is fitted by minimising the negative log-likelihood on a class-balanced set of validation rows disjoint from the evaluation subsample and from the natural-prior sample (at most 2,000 rows per class, seed 7). ECE and its sign are reported before and after scaling on the evaluation subsample and on the natural-prior sample. Predictions, and therefore HSR, escalation, miss and recall, do not change under a temperature.

**Readings, fixed in stage 2 before any result.** Paired ECE (student minus dense) before and after scaling per network and rule, and the fitted temperatures; one expectation stated in advance.

In [ ]:
# Stage 1 - bootstrap (GPU runtime)
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, sys, json, gc, time, copy, itertools, subprocess, importlib
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, yaml
from scipy import stats
import matplotlib
import matplotlib.pyplot as plt

REPO = Path("/content/drive/MyDrive/SABER_Research/saber-ids")
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
importlib.invalidate_caches()
from src.saber.bridge_ciciot import load_bridge
from src.saber.taxonomy import ciciot2023_taxonomy, DEFAULT_COST_PROFILES
from src.saber.metrics import full_model_audit
from src.saber.surgery import prune_cnn1d_channels, count_parameters
from src.saber.deep_model import DeepCNN1D

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
R = REPO / "results/saber"; N40 = R / "40_compression_headroom"
IN46 = R / "46_recovery_budget_curve"; OUT = R / "50_temperature_scaling"; OUT.mkdir(parents=True, exist_ok=True)
MODEL_DIR = REPO / "models/ciciot2023"
assert (N40 / "N40_structures.csv").exists(), "NB40 results missing; pull the repo"
assert (MODEL_DIR / "deepcnn1d_g5_seed0.pt").exists(), "deep teacher checkpoint missing under models/ciciot2023"
assert (IN46 / "N46_runs.csv").exists(), "NB46 results missing; pull the repo"
from scipy.optimize import minimize_scalar
print("device:", DEVICE, "| torch", torch.__version__)

In [ ]:
# Stage 2 - pre-registration (frozen: rerunning this cell can never change it after its commit)
PREREG = {
    "notebook": "50_temperature_scaling",
    "question": ("at 90% on CICIoT2023 after two epochs, does the pruned students' lower calibration error survive when the dense reference "
                 "and every student are each given their own temperature fitted on held-out validation rows"),
    "motivation": ("notebooks 40 and 46: the dense reference's confidence exceeds its accuracy by 0.22 (shallow) and 0.27 (deep) after two "
                   "epochs; students better calibrated in 71 of 80 paired cells. Temperature scaling (Guo et al., 2017) is the standard "
                   "post-hoc fix and a reviewer's first objection"),
    "design": {
        "models": "dense teacher and the committed notebook 40 90% structures for magnitude, Taylor, Fisher; both networks",
        "recovery": "two full epochs, notebook 46's recovery and recalibration code unchanged; seeds 601, 701, 809 (notebook 46's two-epoch seeds)",
        "temperature_fit": ("one scalar T > 0 per recovered and recalibrated model, minimising the mean negative log-likelihood of "
                            "softmax(logits / T) over log T in [log 0.05, log 20] (bounded scalar minimisation), on a fit set of validation "
                            "rows disjoint from the evaluation subsample and the natural-prior sample: at most 2,000 rows per class, seed 7"),
        "evaluation": "notebook 46's class-balanced evaluation subsample (primary) and 200,000-row natural-prior sample; 15-bin equal-mass ECE and mean confidence minus accuracy, before and after scaling",
    },
    "readings": {
        "C_raw": "per network and rule: paired ECE (student minus dense) before scaling, 95% paired t interval; counts better / worse / straddling",
        "C_scaled": "the same after each model's own temperature",
        "T": "fitted temperatures, dense and students, mean and range over seeds",
        "overall": ("advantage_is_temperature if after scaling at most two of the six cells keep the student interval below zero and none "
                    "has it above zero; advantage_survives if at least four of six keep it below zero; students_worse_after_scaling if at "
                    "least two of six have it above zero; mixed otherwise"),
    },
    "expectation": {"T1": "after scaling the dense reference's ECE is at most 0.05 on both networks and the reading is advantage_is_temperature"},
    "reporting_rule": "every reading reported as measured; whichever way it reads, Section IV-D of the paper states it",
    "seeds": [601, 701, 809], "fit_rows_per_class": 2000, "fit_seed": 7, "no_test_access": True, "no_new_selection": True,
}
pre_path = OUT / "N50_PREREGISTRATION.json"
if pre_path.exists():
    print("pre-registration already exists; left untouched:", pre_path)
else:
    pre_path.write_text(json.dumps(PREREG, indent=2)); print("pre-registration written:", pre_path)
print(json.dumps(PREREG, indent=2))

ARCHS = ["shallow", "deep"]; METHODS = ["magnitude", "taylor", "fisher"]; MODELS = ["dense"] + METHODS
SEEDS = PREREG["seeds"]; CAL_SEED = 2026; NAT_SEED = 2026; NAT_ROWS = 200_000
MIN_W = {"shallow": int(yaml.safe_load(open(REPO / "config/saber.yaml"))["groups"]["minimum_remaining_per_layer"]), "deep": 8}
NAME = {"magnitude": "magnitude", "taylor": "Taylor", "fisher": "Fisher", "dense": "dense"}

In [ ]:
# Stage 2b - commit the pre-registration before any result exists (skipped automatically once results exist)
def _prereg_committed():
    rel = "results/saber/50_temperature_scaling/N50_PREREGISTRATION.json"
    h = subprocess.run(["git", "log", "-1", "--format=%h", "--", rel], capture_output=True, text=True, cwd=REPO).stdout.strip()
    dirty = subprocess.run(["git", "status", "--porcelain", "--", rel], capture_output=True, text=True, cwd=REPO).stdout.strip()
    return bool(h) and not dirty


_results_exist = any(p.is_file() and p.name != "N50_PREREGISTRATION.json" for p in OUT.rglob("*"))
if _results_exist:
    assert _prereg_committed(), ("results exist but the pre-registration is not committed, so it can no longer be committed before any "
                                 "result; stop here and report it")
    print("results already exist in", OUT.name, "and the pre-registration is committed; nothing to do here")
else:
    os.chdir(REPO)
    r = subprocess.run(["python", "tools/commit_cell.py",
                        "NB50 pre-registration, committed before any result: does the pruned students' calibration advantage on CICIoT2023 "
                        "survive temperature scaling? Dense teacher and NB40 90% magnitude/Taylor/Fisher structures, both networks, two "
                        "epochs with NB46's code and seeds (601, 701, 809), recalibrated; one temperature per model fitted by NLL on "
                        "class-balanced validation rows disjoint from the evaluation and natural-prior samples (<= 2,000 per class, seed 7); "
                        "paired ECE before and after scaling on both samples. Expectation T1: dense ECE <= 0.05 after scaling and the "
                        "advantage is a temperature effect (<= 2 of 6 cells keep it). No test access, no new selection."],
                       capture_output=True, text=True)
    print(r.stdout); print(r.stderr)

In [ ]:
# Stage 3 - data, teachers and helpers (as notebook 40)
TRAIN_LOADER, VAL_LOADER, _T, SHALLOW_TEACHER, CLASS_NAMES = load_bridge()
taxonomy = ciciot2023_taxonomy(CLASS_NAMES); N_CLASSES = len(CLASS_NAMES)
TEACHERS = {"shallow": SHALLOW_TEACHER.to(DEVICE).eval()}
_dt = DeepCNN1D(N_CLASSES); _dt.load_state_dict(torch.load(MODEL_DIR / "deepcnn1d_g5_seed0.pt", map_location="cpu", weights_only=False)["state_dict"])
TEACHERS["deep"] = _dt.to(DEVICE).eval()

Xv, Yv = VAL_LOADER.dataset.tensors; VAL_Y_ALL = Yv.numpy(); Xt, Yt = TRAIN_LOADER.dataset.tensors; N_TRAIN = len(Xt)
_rng = np.random.default_rng(0)
_idx = np.concatenate([_rng.permutation(np.where(VAL_Y_ALL == c)[0])[:4000] for c in range(N_CLASSES) if (VAL_Y_ALL == c).sum() > 0])
_idx = np.random.default_rng(12345).permutation(_idx)
EX_X = Xv[_idx].to(DEVICE); EX_Y = VAL_Y_ALL[_idx]; EXAMPLE_INPUT = Xv[:8].float().to(DEVICE)
_nat = np.random.default_rng(NAT_SEED).permutation(len(Xv))[:NAT_ROWS]
NAT_X = Xv[_nat].to(DEVICE); NAT_Y = VAL_Y_ALL[_nat]
_counts = np.bincount(Yt.numpy(), minlength=N_CLASSES); _w = np.zeros_like(_counts, dtype=np.float64)
_w[_counts > 0] = 1.0 / np.sqrt(_counts[_counts > 0]); _w[_counts > 0] /= _w[_counts > 0].mean()
CLASS_W = torch.tensor(_w, dtype=torch.float32, device=DEVICE)
_cal_idx = torch.randperm(N_TRAIN, generator=torch.Generator().manual_seed(CAL_SEED))[:50 * 1024]
CAL_LOADER = torch.utils.data.DataLoader(torch.utils.data.Subset(TRAIN_LOADER.dataset, _cal_idx.tolist()), batch_size=1024, shuffle=False)
_sub = Xt[torch.randperm(N_TRAIN, generator=torch.Generator().manual_seed(2026))[:400_000]].numpy()
LO = torch.tensor(np.percentile(_sub, 0.1, axis=0), dtype=torch.float32, device=DEVICE); HI = torch.tensor(np.percentile(_sub, 99.9, axis=0), dtype=torch.float32, device=DEVICE)
del _sub
FAM = np.array(taxonomy.family_by_class); FAMILIES = sorted(set(FAM.tolist())); FAM_Y = FAM[EX_Y]


class Clipped(nn.Module):
    def __init__(self, inner, lo, hi):
        super().__init__(); self.inner = inner; self.register_buffer("lo", lo.clone()); self.register_buffer("hi", hi.clone())
    def forward(self, x):
        return self.inner(torch.maximum(torch.minimum(x, self.hi), self.lo))


def forward_logits(model, X):
    model.eval()
    with torch.no_grad():
        return torch.cat([model(X[i:i + 8192]).cpu() for i in range(0, len(X), 8192)]).numpy()


def forward_logits_batch_stats(model, X):
    saved = {n: (m.running_mean.clone(), m.running_var.clone(), m.momentum, m.num_batches_tracked.clone()) for n, m in model.named_modules() if isinstance(m, nn.BatchNorm1d) and m.running_mean is not None}
    model.train()
    for n, m in model.named_modules():
        if n in saved: m.momentum = 0.0
    with torch.no_grad():
        out = torch.cat([model(X[i:i + 8192]).cpu() for i in range(0, len(X), 8192)]).numpy()
    for n, m in model.named_modules():
        if n in saved:
            rm, rv, mom, nbt = saved[n]; m.running_mean.copy_(rm); m.running_var.copy_(rv); m.momentum = mom; m.num_batches_tracked.copy_(nbt)
    model.eval(); return out


def softmax_np(z):
    z = z - z.max(axis=1, keepdims=True); e = np.exp(z); return e / e.sum(axis=1, keepdims=True)


def audit_logits(lg, y, families=True):
    a = full_model_audit(lg, y, taxonomy, DEFAULT_COST_PROFILES); pr = softmax_np(lg); pred = lg.argmax(axis=1)
    out = {"b2a": float(a["benign_to_attack_rate"]), "a2b": float(a["attack_to_benign_rate"]), "family_f1": float(a["family_macro_f1"]),
           "fine_f1": float(a["fine_macro_f1"]), "hsr_balanced": float(a["hsr_balanced_soc"]), "hsr_miss": float(a["hsr_miss_sensitive"]),
           "hsr_fatigue": float(a["hsr_alert_fatigue"]), "ece": float(a["ece15"]), "conf_minus_acc": float(pr.max(axis=1).mean() - (pred == y).mean())}
    if families:
        pf = FAM[pred]; fy = FAM[y]
        out.update({f"recall_{f}": float((pf[fy == f] == f).mean()) for f in FAMILIES if (fy == f).any()})
    return out


def audit(model, mode="eval"):
    lg = forward_logits(model, EX_X) if mode == "eval" else forward_logits_batch_stats(model, EX_X)
    return audit_logits(lg, EX_Y)


def audit_natural(model):
    return audit_logits(forward_logits(model, NAT_X), NAT_Y, families=False)


def recalibrate_bn(model, n_batches=50):
    for m in model.modules():
        if isinstance(m, nn.BatchNorm1d): m.reset_running_stats(); m.momentum = None
    model.train()
    with torch.no_grad():
        for i, (xb, _) in enumerate(CAL_LOADER):
            if i >= n_batches: break
            model(xb.to(DEVICE))
    for m in model.modules():
        if isinstance(m, nn.BatchNorm1d): m.momentum = 0.1
    model.eval(); return model


def make_loader(seed, fraction):
    gen = torch.Generator().manual_seed(seed)
    if fraction < 1.0:
        sub = torch.randperm(N_TRAIN, generator=gen)[: int(N_TRAIN * fraction)]
        return torch.utils.data.DataLoader(torch.utils.data.Subset(TRAIN_LOADER.dataset, sub.tolist()), batch_size=1024, shuffle=True, generator=torch.Generator().manual_seed(seed))
    return torch.utils.data.DataLoader(TRAIN_LOADER.dataset, batch_size=1024, shuffle=True, generator=torch.Generator().manual_seed(seed))


def recover(model, loader, epochs):
    opt = torch.optim.Adam(model.parameters(), lr=1e-3); lossf = nn.CrossEntropyLoss(weight=CLASS_W)
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            opt.zero_grad(); lossf(model(xb.to(DEVICE)), yb.to(DEVICE)).backward(); opt.step()
    return model.eval()


def prune_map_of(pairs):
    pm = {}
    for layer, ch in pairs: pm.setdefault(str(layer), []).append(int(ch))
    return {l: sorted(c) for l, c in pm.items()}


def student_for(arch, method):
    if method == "dense":
        return copy.deepcopy(TEACHERS[arch]).to(DEVICE)
    rm = pd.read_csv(N40 / f"N40_{arch}_{method}_b90_removed_groups.csv")
    st, _ = prune_cnn1d_channels(TEACHERS[arch], prune_map_of(zip(rm.module_path.astype(str), rm.channel_index.astype(int))), EXAMPLE_INPUT, minimum_remaining_per_layer=MIN_W[arch])
    return st.to(DEVICE)


STRUCT40 = pd.read_csv(N40 / "N40_structures.csv").set_index(["architecture", "method", "budget"])
for arch in ARCHS:
    for m in METHODS:
        n = count_parameters(student_for(arch, m)); ref = int(STRUCT40.loc[(arch, m, 0.9), "parameters"])
        assert n == ref, f"{arch}/{m}: rebuilt 90% structure has {n} parameters, NB40 recorded {ref}"
print("CICIoT2023 ready | train", N_TRAIN, "| balanced eval rows", len(_idx), "| natural-prior rows", len(_nat), "| 90% structures match N40_structures.csv")

In [ ]:
# Stage 4 - temperature fit set (validation rows disjoint from both evaluation samples) and the scaling helpers
_used = np.zeros(len(VAL_Y_ALL), dtype=bool); _used[_idx] = True; _used[_nat] = True
_rng7 = np.random.default_rng(PREREG["fit_seed"])
_fit = np.concatenate([_rng7.permutation(np.where((VAL_Y_ALL == c) & ~_used)[0])[:PREREG["fit_rows_per_class"]] for c in range(N_CLASSES)])
assert not np.intersect1d(_fit, _idx).size and not np.intersect1d(_fit, _nat).size, "fit rows overlap an evaluation sample"
assert len(_fit) >= 10_000, f"only {len(_fit)} validation rows remain for the temperature fit"
FIT_X = Xv[_fit].to(DEVICE); FIT_Y = VAL_Y_ALL[_fit]
print("temperature fit rows:", len(_fit), "| classes represented:", len(np.unique(FIT_Y)), "of", N_CLASSES)


def nll(lg, y, T):
    z = lg / T; z = z - z.max(axis=1, keepdims=True); lse = np.log(np.exp(z).sum(axis=1))
    return float((lse - z[np.arange(len(y)), y]).mean())


def fit_temperature(lg, y):
    res = minimize_scalar(lambda lt: nll(lg, y, np.exp(lt)), bounds=(np.log(0.05), np.log(20.0)), method="bounded", options={"xatol": 1e-4})
    return float(np.exp(res.x))

In [ ]:
# Stage 5 - the runs: 2 networks x (dense + 3 rules) x 3 seeds, two epochs, then one temperature per model (resumable)
RUNS = OUT / "N50_runs.csv"
rows = pd.read_csv(RUNS).to_dict("records") if RUNS.exists() else []
done = {(r["architecture"], r["method"], int(r["seed"])) for r in rows}


def run_one(arch, method, seed):
    # notebook 46's run_one at budget 2.00, then the temperature
    torch.manual_seed(seed); np.random.seed(seed); t0 = time.time()
    model = Clipped(student_for(arch, method), LO, HI).to(DEVICE)
    recover(model, make_loader(seed, 1.0), 2); recalibrate_bn(model)
    lg_ex, lg_nat, lg_fit = forward_logits(model, EX_X), forward_logits(model, NAT_X), forward_logits(model, FIT_X)
    T = fit_temperature(lg_fit, FIT_Y)
    raw, nat = audit_logits(lg_ex, EX_Y), audit_logits(lg_nat, NAT_Y, families=False)
    sc, nsc = audit_logits(lg_ex / T, EX_Y), audit_logits(lg_nat / T, NAT_Y, families=False)
    assert abs(raw["hsr_balanced"] - sc["hsr_balanced"]) < 1e-9, "a temperature changed the predictions"
    row = {"architecture": arch, "method": method, "seed": seed, "T": T, "fit_nll_raw": nll(lg_fit, FIT_Y, 1.0), "fit_nll_scaled": nll(lg_fit, FIT_Y, T),
           "hsr_balanced": raw["hsr_balanced"], "b2a": raw["b2a"], "fine_f1": raw["fine_f1"],
           "ece_raw": raw["ece"], "cma_raw": raw["conf_minus_acc"], "ece_scaled": sc["ece"], "cma_scaled": sc["conf_minus_acc"],
           "nat_ece_raw": nat["ece"], "nat_cma_raw": nat["conf_minus_acc"], "nat_ece_scaled": nsc["ece"], "nat_cma_scaled": nsc["conf_minus_acc"],
           "seconds": round(time.time() - t0, 1)}
    del model; gc.collect()
    return row


JOBS = [(a, m, s) for a in ARCHS for s in SEEDS for m in MODELS]
for job in JOBS:
    if (job[0], job[1], int(job[2])) in done: continue
    row = run_one(*job); rows.append(row); done.add((job[0], job[1], int(job[2]))); pd.DataFrame(rows).to_csv(RUNS, index=False)
    print(f"{job[0]:7s} {NAME[job[1]]:9s} s{job[2]}: T {row['T']:.3f} | ECE {row['ece_raw']:.3f} -> {row['ece_scaled']:.3f} (conf-acc {row['cma_raw']:+.3f} -> {row['cma_scaled']:+.3f}) | "
          f"natural {row['nat_ece_raw']:.3f} -> {row['nat_ece_scaled']:.3f} | HSR {row['hsr_balanced']:.4f} | {row['seconds']:.0f}s")
print("runs:", len(rows), "of", len(JOBS))

In [ ]:
# Stage 6 - analysis and verdict (rules exactly as pre-registered in stage 2)
runs = pd.read_csv(RUNS)
assert len(runs) == len(JOBS), f"expected {len(JOBS)} runs, found {len(runs)}: finish stage 5 first"


def paired(arch, method, col):
    s = runs[(runs.architecture == arch) & (runs.method == method)].set_index("seed")[col]; d = runs[(runs.architecture == arch) & (runs.method == "dense")].set_index("seed")[col]
    c = s.index.intersection(d.index); g = (s.loc[c] - d.loc[c]).to_numpy(); n = len(g); hw = stats.t.ppf(0.975, n - 1) * g.std(ddof=1) / np.sqrt(n)
    return {"gap": float(g.mean()), "lo": float(g.mean() - hw), "hi": float(g.mean() + hw)}


out = []
for arch in ARCHS:
    d = runs[(runs.architecture == arch) & (runs.method == "dense")]
    out.append({"architecture": arch, "method": "dense", "T": float(d["T"].mean()), "T_min": float(d["T"].min()), "T_max": float(d["T"].max()),
                **{c: float(d[c].mean()) for c in ("ece_raw", "ece_scaled", "cma_raw", "cma_scaled", "nat_ece_raw", "nat_ece_scaled")}})
    for m in METHODS:
        q = runs[(runs.architecture == arch) & (runs.method == m)]
        r_, s_, nr, ns_ = paired(arch, m, "ece_raw"), paired(arch, m, "ece_scaled"), paired(arch, m, "nat_ece_raw"), paired(arch, m, "nat_ece_scaled")
        out.append({"architecture": arch, "method": m, "T": float(q["T"].mean()), "T_min": float(q["T"].min()), "T_max": float(q["T"].max()),
                    **{c: float(q[c].mean()) for c in ("ece_raw", "ece_scaled", "cma_raw", "cma_scaled", "nat_ece_raw", "nat_ece_scaled")},
                    "gap_raw": r_["gap"], "gap_raw_lo": r_["lo"], "gap_raw_hi": r_["hi"], "gap_scaled": s_["gap"], "gap_scaled_lo": s_["lo"], "gap_scaled_hi": s_["hi"],
                    "nat_gap_raw": nr["gap"], "nat_gap_raw_lo": nr["lo"], "nat_gap_raw_hi": nr["hi"], "nat_gap_scaled": ns_["gap"], "nat_gap_scaled_lo": ns_["lo"], "nat_gap_scaled_hi": ns_["hi"]})
SUM = pd.DataFrame(out); SUM.to_csv(OUT / "N50_summary.csv", index=False)
st = SUM[SUM.method != "dense"]
counts = {"raw_better": int((st.gap_raw_hi < 0).sum()), "raw_worse": int((st.gap_raw_lo > 0).sum()),
          "scaled_better": int((st.gap_scaled_hi < 0).sum()), "scaled_worse": int((st.gap_scaled_lo > 0).sum()),
          "nat_raw_better": int((st.nat_gap_raw_hi < 0).sum()), "nat_scaled_better": int((st.nat_gap_scaled_hi < 0).sum()), "nat_scaled_worse": int((st.nat_gap_scaled_lo > 0).sum()), "cells": int(len(st))}
overall = ("students_worse_after_scaling" if counts["scaled_worse"] >= 2 else
           "advantage_is_temperature" if (counts["scaled_better"] <= 2 and counts["scaled_worse"] == 0) else
           "advantage_survives" if counts["scaled_better"] >= 4 else "mixed")
dense_scaled = {a: round(float(SUM[(SUM.architecture == a) & (SUM.method == "dense")].ece_scaled.iloc[0]), 4) for a in ARCHS}
T1 = bool(all(v <= 0.05 for v in dense_scaled.values()) and overall == "advantage_is_temperature")
# replication check against notebook 46's two-epoch rows (same code and seeds; GPU kernels are not bit-deterministic)
n46 = pd.read_csv(IN46 / "N46_runs.csv"); n46 = n46[np.isclose(n46.budget_r, 2.0)][["architecture", "method", "seed", "recal_hsr_balanced", "recal_ece"]]
rep = runs.merge(n46, on=["architecture", "method", "seed"], how="left")
replication = {"max_abs_hsr_diff": float((rep.hsr_balanced - rep.recal_hsr_balanced).abs().max()), "max_abs_ece_diff": float((rep.ece_raw - rep.recal_ece).abs().max())}
verdict = {"notebook": "50_temperature_scaling", "overall": overall, "counts": counts, "dense_ece_scaled": dense_scaled, "T1": T1, "replication_vs_nb46": replication,
           "summary": SUM.round(4).to_dict("records"), "prereg": PREREG}
(OUT / "N50_verdict.json").write_text(json.dumps(verdict, indent=2, default=float))
print("OVERALL:", overall, "| T1", T1, "| counts", counts, "| dense ECE after scaling", dense_scaled, "| replication vs NB46", replication)
print(SUM[["architecture", "method", "T", "T_min", "T_max", "ece_raw", "ece_scaled", "cma_raw", "cma_scaled", "gap_raw", "gap_raw_lo", "gap_raw_hi", "gap_scaled", "gap_scaled_lo", "gap_scaled_hi", "nat_ece_scaled"]].round(4).to_string(index=False))

In [ ]:
# Stage 7 - figure (600 dpi, paper names): ECE before and after temperature scaling, dense and students
plt.rcParams.update({"font.family": "serif", "font.serif": ["Liberation Serif", "DejaVu Serif"], "font.size": 7, "axes.labelsize": 7,
                     "xtick.labelsize": 6.5, "ytick.labelsize": 6.5, "legend.fontsize": 6, "pdf.fonttype": 42, "axes.linewidth": 0.6})
COLR = {"magnitude": "#E69F00", "taylor": "#56B4E9", "fisher": "#009E73", "dense": "#1F1F1F"}
fig, axes = plt.subplots(1, 2, figsize=(3.5, 1.7), sharey=True, gridspec_kw={"wspace": 0.08})
for ax, arch in zip(axes, ARCHS):
    s = SUM[SUM.architecture == arch].set_index("method").loc[MODELS]; x = np.arange(len(MODELS)); w = 0.36
    ax.bar(x - w / 2, s.ece_raw, w, color=[COLR[m] for m in MODELS], alpha=0.45, label="as recovered")
    ax.bar(x + w / 2, s.ece_scaled, w, color=[COLR[m] for m in MODELS], label="temperature scaled")
    ax.set_xticks(x); ax.set_xticklabels([NAME[m] for m in MODELS], rotation=30, ha="right"); ax.set_title(f"{arch} network", fontsize=7, pad=2)
    ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y", color="#E6E6E6", lw=0.5); ax.set_axisbelow(True)
axes[0].set_ylabel("ECE"); axes[0].legend(frameon=False, loc="upper right", handlelength=1.0)
fig.savefig(OUT / "N50_temperature.png", dpi=600, bbox_inches="tight"); fig.savefig(OUT / "N50_temperature.pdf", bbox_inches="tight"); plt.show()
print("written ->", OUT)

In [ ]:
# Stage 8 - commit the results (standard commit cell; the message is filled in from the verdict)
v = json.load(open(OUT / "N50_verdict.json"))
pre = subprocess.run(["git", "log", "-1", "--format=%h", "--", "results/saber/50_temperature_scaling/N50_PREREGISTRATION.json"], capture_output=True, text=True, cwd=REPO).stdout.strip() or "see log"
txt = "; ".join(f"{r['architecture']}/{NAME[r['method']]} T {r['T']:.2f}, ECE {r['ece_raw']:.3f} -> {r['ece_scaled']:.3f}"
                + (f", paired gap {r['gap_raw']:+.3f} -> {r['gap_scaled']:+.3f} [{r['gap_scaled_lo']:+.3f}, {r['gap_scaled_hi']:+.3f}]" if r["method"] != "dense" else "") for r in v["summary"])
c = v["counts"]
msg = (f"NB50 results (pre-registration {pre}): temperature scaling at 90% on CICIoT2023 after two epochs (seeds 601/701/809, 24 models). Overall {v['overall']}; "
       f"students better before scaling in {c['raw_better']} of {c['cells']} cells, after scaling better in {c['scaled_better']} / worse in {c['scaled_worse']}; "
       f"dense ECE after scaling {v['dense_ece_scaled']}; T1 {'HELD' if v['T1'] else 'NOT HELD'}. {txt}. Replication vs NB46: max |HSR diff| {v['replication_vs_nb46']['max_abs_hsr_diff']:.4f}, max |ECE diff| {v['replication_vs_nb46']['max_abs_ece_diff']:.4f}.")
print(msg)
os.chdir(REPO)
r = subprocess.run(["python", "tools/commit_cell.py", msg], capture_output=True, text=True)
print(r.stdout); print(r.stderr)